## 📦 Part 0: 환경 설정 및 라이브러리 설치

필요한 패키지들을 설치하고 임포트합니다.

In [1]:
# 필수 패키지 설치
!pip install -q langchain langgraph langchain-openai duckduckgo-search wikipedia-api


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 라이브러리 임포트
import os
from typing import TypedDict, Dict, List, Any, Optional, Annotated, Sequence, Literal
from datetime import datetime
import json
import re
import operator

# LangChain 핵심 컴포넌트
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# LangGraph 컴포넌트
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 도구들
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import Tool

# 시각화
from IPython.display import Image, display
import matplotlib.pyplot as plt

#ydantic 관련 경고 메시지 숨기기
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Pydantic.*")

print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다!")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다!


## 🔑 Part 1: API 키 설정 및 LLM 초기화

In [3]:
from dotenv import load_dotenv

load_dotenv()

# OpenAI API 키 확인
if not os.getenv("OPENAI_API_KEY"):
    print(" * 경고: OPENAI_API_KEY가 .env 파일에 설정되지 않았습니다.")
else:
    # API 키의 처음 10자만 표시 (보안)
    api_key = os.getenv("OPENAI_API_KEY")
    print(f" * OpenAI API 키 로드 완료 !")

 * OpenAI API 키 로드 완료 !


In [4]:
# LLM 초기화
llm = ChatOpenAI(model="gpt-5-mini")

# 비평용 LLM 
critic_llm = ChatOpenAI(model="gpt-5-mini")

print("✅ LLM이 초기화되었습니다!")

✅ LLM이 초기화되었습니다!


## 🏗️ Part 2: 상태(State) 정의

다중 에이전트 시스템이 공유할 상태를 정의합니다.

In [5]:
class MultiAgentState(TypedDict):
    """
    다중 에이전트 시스템의 상태를 정의합니다.
    각 에이전트가 공유하는 정보를 관리합니다.
    """

    messages: Sequence[BaseMessage]  # 대화 히스토리
    current_agent: str  # 현재 작업 중인 에이전트
    task: str  # 수행할 작업
    task_queue: List[Dict]  # 작업 큐
    agent_results: Dict[str, Any]  # 각 에이전트의 결과
    reflection_count: Dict[str, int]  # 에이전트별 성찰 횟수
    quality_scores: Dict[str, float]  # 품질 점수
    final_output: Optional[str]  # 최종 출력
    iteration: int  # 전체 반복 횟수
    max_iterations: int  # 최대 반복 횟수


print("✅ State 정의 완료!")

✅ State 정의 완료!


## 🛠️ Part 3: 도구(Tools) 설정

에이전트들이 사용할 웹 검색과 Wikipedia 도구를 설정합니다.

In [6]:
# 웹 검색 도구 설정
search_tool = DuckDuckGoSearchRun()
search = Tool(
    name="web_search",
    description="Search the web for current information",
    func=search_tool.run,
)

# Wikipedia 도구 설정
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki_tool = Tool(
    name="wikipedia",
    description="Search Wikipedia for detailed information",
    func=wikipedia.run,
)

tools = [search, wiki_tool]
print(f"✅ {len(tools)}개의 도구가 준비되었습니다!")

✅ 2개의 도구가 준비되었습니다!


## 👥 Part 4: 전문 에이전트 클래스 정의

### 4.1 Research Agent (연구원 에이전트) 🔍

In [7]:
class ResearchAgent:
    """
    🔍 연구원 에이전트
    - 웹과 Wikipedia에서 정보를 검색하고 분석합니다
    - 최신 트렌드와 신뢰할 수 있는 정보를 수집합니다
    """

    def __init__(self):
        self.name = "Research Agent"
        self.llm = llm
        self.search = search_tool
        self.wikipedia = wikipedia

    def perform_research(self, task: str) -> Dict:
        """주어진 주제에 대해 연구를 수행합니다"""

        print(f"\n🔍 {self.name} 작업 시작...")

        # 1. 웹 검색
        search_query = f"{task} 2024 2025 latest trends"
        print(f"  검색 중: {search_query}")
        try:
            search_results = self.search.run(search_query)
        except:
            search_results = "웹 검색 결과를 가져올 수 없습니다."

        # 2. Wikipedia 검색
        wiki_query = task.split()[0] if task else "AI"
        try:
            wiki_results = self.wikipedia.run(wiki_query)[:500]
        except:
            wiki_results = "Wikipedia 정보를 찾을 수 없습니다."

        # 3. LLM으로 정보 종합
        synthesis_prompt = f"""
        다음 정보를 바탕으로 '{task}'에 대한 핵심 내용을 정리하세요:
        
        웹 검색 결과:
        {search_results[:1000]}
        
        Wikipedia:
        {wiki_results}
        
        다음 형식으로 정리하세요:
        1. 핵심 발견사항 (3-5개)
        2. 최신 트렌드
        3. 중요한 통계나 수치
        4. 신뢰할 수 있는 출처
        """

        response = self.llm.invoke(synthesis_prompt)

        return {
            "agent": self.name,
            "content": response.content,
            "sources": {
                "web_search": search_results[:200] + "...",
                "wikipedia": wiki_results[:200] + "...",
            },
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    def self_evaluate(self, result: Dict) -> float:
        """자신의 연구 결과를 평가합니다"""

        eval_prompt = f"""
        다음 연구 결과의 품질을 1-10점 척도로 평가하세요.
        
        연구 결과:
        {result['content'][:800]}
        
        평가 기준:
        - 정보의 최신성 (2024-2025년 정보 포함 여부)
        - 출처의 신뢰성
        - 내용의 깊이와 포괄성
        - 구체적인 수치나 통계 포함 여부
        
        점수만 숫자로 답하세요. (예: 7.5)
        """

        score_response = self.llm.invoke(eval_prompt)
        try:
            score = float(score_response.content.strip())
            score = min(max(score, 1.0), 10.0)
        except:
            score = 5.0

        return score

    def improve(self, result: Dict, feedback: str) -> Dict:
        """피드백을 바탕으로 연구 결과를 개선합니다"""

        improve_prompt = f"""
        현재 연구 결과를 다음 피드백을 바탕으로 개선하세요:
        
        현재 결과:
        {result['content']}
        
        피드백:
        {feedback}
        
        개선된 연구 결과를 작성하세요. 특히:
        - 더 구체적인 수치와 통계 추가
        - 최신 사례나 트렌드 포함
        - 신뢰할 수 있는 기관/전문가 인용
        """

        improved_response = self.llm.invoke(improve_prompt)
        result["content"] = improved_response.content
        result["improved"] = True
        result["improvement_feedback"] = feedback

        return result


print("✅ Research Agent 정의 완료!")

✅ Research Agent 정의 완료!


### 4.2 Writer Agent (작가 에이전트) ✍️

In [8]:
class WriterAgent:
    """
    ✍️ 작가 에이전트
    - 구조화된 콘텐츠를 작성합니다
    - 읽기 쉽고 전문적인 문체를 유지합니다
    """

    def __init__(self):
        self.name = "Writer Agent"
        self.llm = llm

    def write_content(self, task: str, research_data: Optional[Dict] = None) -> Dict:
        """주어진 주제로 콘텐츠를 작성합니다"""

        print(f"\n✍️ {self.name} 작업 시작...")

        # 연구 자료가 있으면 포함
        context = ""
        if research_data and "content" in research_data:
            context = f"\n[연구 자료]\n{research_data['content']}\n"

        write_prompt = f"""
        다음 주제에 대한 고품질 콘텐츠를 작성하세요: {task}
        {context}
        
        요구사항:
        1. 명확한 구조 (서론-본론-결론)
        2. 독자 친화적인 문체
        3. 구체적인 예시와 비유 포함
        4. 실용적인 인사이트 제공
        5. 전문적이면서도 이해하기 쉬운 설명
        
        작성 스타일: 전문적이면서 친근한 톤
        대상 독자: 일반 독자부터 전문가까지
        """

        response = self.llm.invoke(write_prompt)

        return {
            "agent": self.name,
            "content": response.content,
            "structure": {"introduction": True, "body": True, "conclusion": True},
            "style": "professional-friendly",
            "based_on_research": research_data is not None,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    def self_evaluate(self, result: Dict) -> float:
        """작성한 콘텐츠를 자가 평가합니다"""

        eval_prompt = f"""
        다음 글의 품질을 1-10점 척도로 평가하세요.
        
        작성된 글:
        {result['content'][:800]}
        
        평가 기준:
        - 구조의 명확성 (서론-본론-결론)
        - 가독성과 문체의 일관성
        - 정보의 유용성과 실용성
        - 예시와 설명의 구체성
        
        점수만 숫자로 답하세요. (예: 8.0)
        """

        score_response = self.llm.invoke(eval_prompt)
        try:
            score = float(score_response.content.strip())
            score = min(max(score, 1.0), 10.0)
        except:
            score = 5.0

        return score

    def improve(self, result: Dict, feedback: str) -> Dict:
        """피드백을 바탕으로 글을 개선합니다"""

        improve_prompt = f"""
        현재 글을 다음 피드백을 바탕으로 개선하세요:
        
        현재 글:
        {result['content']}
        
        피드백:
        {feedback}
        
        개선 사항:
        - 구조를 더 명확하게
        - 구체적인 예시 추가
        - 문장을 더 간결하고 명료하게
        - 핵심 메시지를 더 강조
        """

        improved_response = self.llm.invoke(improve_prompt)
        result["content"] = improved_response.content
        result["improved"] = True
        result["improvement_feedback"] = feedback

        return result


print("✅ Writer Agent 정의 완료!")

✅ Writer Agent 정의 완료!


### 4.3 Critic Agent (비평가 에이전트) 🎯

In [9]:
class CriticAgent:
    """
    🎯 비평가 에이전트
    - 다른 에이전트의 결과물을 평가합니다
    - 구체적인 개선점을 제시합니다
    """

    def __init__(self):
        self.name = "Critic Agent"
        self.llm = critic_llm  # 더 엄격한 평가를 위해 낮은 temperature

    def evaluate(self, content: Dict, task: str) -> Dict:
        """콘텐츠를 평가하고 피드백을 제공합니다"""

        print(f"\n🎯 {self.name} 평가 시작...")

        eval_prompt = f"""
        다음 작업 결과를 상세히 평가해주세요.
        
        원래 작업: {task}
        
        평가할 콘텐츠:
        {content.get('content', '')[:1500]}
        
        다음 기준으로 평가하세요:
        1. 정확성 (1-10점): 정보의 정확도와 신뢰성
        2. 완성도 (1-10점): 작업 요구사항 충족도
        3. 가독성 (1-10점): 구조와 문체의 명확성
        4. 창의성 (1-10점): 독창적 인사이트와 접근법
        
        형식:
        ## 점수
        - 정확성: X/10
        - 완성도: X/10
        - 가독성: X/10
        - 창의성: X/10
        - 종합 점수: X/10
        
        ## 잘한 점 (최소 2개)
        
        ## 개선이 필요한 점 (최소 3개)
        
        ## 구체적인 개선 제안
        """

        response = self.llm.invoke(eval_prompt)

        # 점수 파싱
        scores = self._parse_scores(response.content)

        return {
            "agent": self.name,
            "evaluation": response.content,
            "scores": scores,
            "average_score": sum(scores.values()) / len(scores) if scores else 5.0,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    def _parse_scores(self, text: str) -> Dict[str, float]:
        """평가 텍스트에서 점수를 추출합니다"""

        scores = {}

        # 정규 표현식으로 점수 추출
        patterns = {
            "accuracy": r"정확성[:\s]*(\d+(?:\.\d+)?)",
            "completeness": r"완성도[:\s]*(\d+(?:\.\d+)?)",
            "readability": r"가독성[:\s]*(\d+(?:\.\d+)?)",
            "creativity": r"창의성[:\s]*(\d+(?:\.\d+)?)",
            "overall": r"종합\s*점수[:\s]*(\d+(?:\.\d+)?)",
        }

        for key, pattern in patterns.items():
            match = re.search(pattern, text)
            if match:
                scores[key] = float(match.group(1))
            else:
                scores[key] = 5.0  # 기본값

        return scores

    def generate_feedback(self, scores: Dict[str, float]) -> str:
        """점수를 바탕으로 개선 피드백을 생성합니다"""

        feedback_parts = []

        if scores.get("accuracy", 10) < 7:
            feedback_parts.append("정보의 정확성과 출처 신뢰도를 높이세요")

        if scores.get("completeness", 10) < 7:
            feedback_parts.append("작업 요구사항을 더 충실히 반영하세요")

        if scores.get("readability", 10) < 7:
            feedback_parts.append("문장 구조와 가독성을 개선하세요")

        if scores.get("creativity", 10) < 7:
            feedback_parts.append("더 독창적인 관점과 예시를 추가하세요")

        return " | ".join(feedback_parts) if feedback_parts else "전반적으로 우수합니다"


print("✅ Critic Agent 정의 완료!")

✅ Critic Agent 정의 완료!


### 4.4 Code Agent (코드 에이전트) 💻

In [10]:
class CodeAgent:
    """
    💻 코드 에이전트
    - 요구사항에 맞는 코드를 생성합니다
    - 문서화와 주석을 포함합니다
    """

    def __init__(self):
        self.name = "Code Agent"
        self.llm = llm

    def generate_code(self, task: str, context: Optional[Dict] = None) -> Dict:
        """요구사항에 맞는 코드를 생성합니다"""

        print(f"\n💻 {self.name} 작업 시작...")

        code_prompt = f"""
        다음 요구사항에 맞는 Python 코드를 작성하세요: {task}
        
        코드 작성 규칙:
        1. 깔끔하고 읽기 쉬운 코드
        2. 함수/클래스로 잘 구조화
        3. 상세한 docstring과 주석
        4. 에러 처리 포함
        5. 타입 힌트 사용
        6. 사용 예시 포함
        
        형식:
        ```python
        # 코드 설명
        
        # 필요한 임포트
        
        # 메인 코드
        
        # 사용 예시
        if __name__ == "__main__":
            # 예시 코드
        ```
        """

        response = self.llm.invoke(code_prompt)

        return {
            "agent": self.name,
            "content": response.content,
            "type": "code",
            "language": "python",
            "has_examples": True,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    def self_evaluate(self, result: Dict) -> float:
        """생성한 코드를 평가합니다"""

        eval_prompt = f"""
        다음 코드의 품질을 1-10점 척도로 평가하세요.
        
        코드:
        {result['content'][:1000]}
        
        평가 기준:
        - 코드의 정확성과 동작 여부
        - 가독성과 구조화
        - 주석과 문서화의 충실도
        - 에러 처리와 안정성
        - 코드 스타일과 관례 준수
        
        점수만 숫자로 답하세요. (예: 7.5)
        """

        score_response = self.llm.invoke(eval_prompt)
        try:
            score = float(score_response.content.strip())
            score = min(max(score, 1.0), 10.0)
        except:
            score = 5.0

        return score

    def improve(self, result: Dict, feedback: str) -> Dict:
        """피드백을 바탕으로 코드를 개선합니다"""

        improve_prompt = f"""
        현재 코드를 다음 피드백을 바탕으로 개선하세요:
        
        현재 코드:
        {result['content']}
        
        피드백:
        {feedback}
        
        개선 사항:
        - 더 상세한 주석 추가
        - 에러 처리 강화
        - 코드 구조 개선
        - 성능 최적화
        - 더 많은 사용 예시
        """

        improved_response = self.llm.invoke(improve_prompt)
        result["content"] = improved_response.content
        result["improved"] = True
        result["improvement_feedback"] = feedback

        return result


print("✅ Code Agent 정의 완료!")
print("\n📊 정의된 에이전트들:")
print("  1. Research Agent - 정보 검색 및 분석")
print("  2. Writer Agent - 콘텐츠 작성")
print("  3. Critic Agent - 평가 및 피드백")
print("  4. Code Agent - 코드 생성")

✅ Code Agent 정의 완료!

📊 정의된 에이전트들:
  1. Research Agent - 정보 검색 및 분석
  2. Writer Agent - 콘텐츠 작성
  3. Critic Agent - 평가 및 피드백
  4. Code Agent - 코드 생성


## 🔄 Part 5: 성찰(Reflection) 메커니즘 구현

에이전트가 자신의 작업을 평가하고 개선하는 핵심 메커니즘입니다.

In [11]:
def reflect_and_improve(agent: Any, result: Dict, state: MultiAgentState) -> Dict:
    """
    🔄 에이전트의 성찰 및 개선 메커니즘

    작동 방식:
    1. 자가 평가 수행 (1-10점)
    2. 7점 미만이면 개선 수행
    3. 최대 3회까지 반복
    """

    agent_name = agent.name

    # 반복 횟수 초기화
    if agent_name not in state["reflection_count"]:
        state["reflection_count"][agent_name] = 0

    reflection_count = state["reflection_count"][agent_name]

    # 최대 3회 반복
    if reflection_count >= 3:
        print(f"  ⚠️ {agent_name}: 최대 반복 횟수 도달")
        return result

    # 자가 평가
    score = agent.self_evaluate(result)
    state["quality_scores"][agent_name] = score

    print(f"  📊 {agent_name} 자가 평가 점수: {score:.1f}/10")

    # 7점 미만이면 개선
    if score < 7.0:
        # 피드백 생성
        if hasattr(agent, "generate_feedback"):
            feedback = agent.generate_feedback(result.get("scores", {}))
        else:
            feedback = f"""
            현재 점수가 {score:.1f}/10입니다. 다음을 개선하세요:
            - 더 구체적이고 상세한 내용 추가
            - 구조와 가독성 향상
            - 실용적인 예시와 인사이트 포함
            - 최신 정보와 트렌드 반영
            """

        print(f"  🔧 {agent_name} 개선 중... (시도 {reflection_count + 1}/3)")

        # 개선 수행
        improved_result = agent.improve(result, feedback)

        # 반복 횟수 증가
        state["reflection_count"][agent_name] = reflection_count + 1

        # 재귀적으로 다시 평가
        return reflect_and_improve(agent, improved_result, state)

    print(f"  ✅ {agent_name}: 품질 기준 통과!")
    return result


print("✅ 성찰 메커니즘 구현 완료!")
print("\n💡 성찰 메커니즘 특징:")
print("  • 자동 품질 평가 (1-10점)")
print("  • 7점 미만 시 자동 개선")
print("  • 최대 3회 반복으로 품질 향상")

✅ 성찰 메커니즘 구현 완료!

💡 성찰 메커니즘 특징:
  • 자동 품질 평가 (1-10점)
  • 7점 미만 시 자동 개선
  • 최대 3회 반복으로 품질 향상


## 👔 Part 6: 감독관(Supervisor) 구현

전체 작업을 관리하고 조율하는 중앙 컨트롤러입니다.

In [12]:
class Supervisor:
    """
    👔 감독관
    - 전체 작업을 관리하고 조율합니다
    - 에이전트에게 작업을 할당합니다
    - 최종 결과를 통합합니다
    """

    def __init__(self):
        self.name = "Supervisor"
        self.llm = llm

    def analyze_task(self, task: str) -> List[str]:
        """작업을 분석하고 필요한 에이전트를 결정합니다"""

        print(f"\n👔 Supervisor: 작업 분석 중...")

        # 작업 유형별 에이전트 시퀀스 결정
        task_lower = task.lower()

        # 코드 관련 작업
        if any(
            keyword in task_lower
            for keyword in ["코드", "프로그램", "스크립트", "함수", "클래스"]
        ):
            sequence = ["Research Agent", "Code Agent", "Critic Agent"]

        # 분석/조사 작업
        elif any(
            keyword in task_lower for keyword in ["분석", "조사", "연구", "트렌드"]
        ):
            sequence = [
                "Research Agent",
                "Writer Agent",
                "Critic Agent",
                "Writer Agent",
            ]

        # 창작/작성 작업
        elif any(
            keyword in task_lower
            for keyword in ["작성", "글", "에세이", "보고서", "문서"]
        ):
            sequence = ["Research Agent", "Writer Agent", "Critic Agent"]

        # 튜토리얼/가이드
        elif any(
            keyword in task_lower for keyword in ["튜토리얼", "가이드", "설명", "교육"]
        ):
            sequence = ["Research Agent", "Writer Agent", "Code Agent", "Critic Agent"]

        # 기본 시퀀스
        else:
            sequence = ["Research Agent", "Writer Agent", "Critic Agent"]

        print(f"  📋 선택된 에이전트 순서: {' → '.join(sequence)}")
        return sequence

    def integrate_results(self, results: Dict[str, Any], task: str) -> str:
        """모든 에이전트의 결과를 통합합니다"""

        print(f"\n👔 Supervisor: 결과 통합 중...")

        # 결과 요약
        results_summary = ""
        for agent_name, result in results.items():
            if isinstance(result, dict) and "content" in result:
                results_summary += f"\n[{agent_name}]\n{result['content'][:500]}...\n"

        integration_prompt = f"""
        다음 에이전트들의 작업 결과를 하나의 완성된 문서로 통합하세요.
        
        원래 작업: {task}
        
        에이전트 결과물:
        {results_summary}
        
        통합 규칙:
        1. 모든 중요한 정보를 포함
        2. 중복 내용 제거
        3. 일관된 흐름과 구조 유지
        4. 전문적이고 완성도 높은 마무리
        5. 핵심 인사이트 강조
        
        최종 문서를 작성하세요:
        """

        response = self.llm.invoke(integration_prompt)
        return response.content

    def generate_summary(self, state: MultiAgentState) -> str:
        """작업 수행 과정을 요약합니다"""

        summary = f"""
        📊 작업 수행 요약
        ================
        작업: {state['task']}
        참여 에이전트: {len(state['agent_results'])}개
        총 반복 횟수: {state['iteration']}회
        
        품질 점수:
        """

        for agent, score in state["quality_scores"].items():
            summary += f"\n  • {agent}: {score:.1f}/10"
            if agent in state["reflection_count"]:
                summary += f" (개선 {state['reflection_count'][agent]}회)"

        avg_score = (
            sum(state["quality_scores"].values()) / len(state["quality_scores"])
            if state["quality_scores"]
            else 0
        )
        summary += f"\n\n평균 점수: {avg_score:.1f}/10"

        return summary


print("✅ Supervisor 클래스 정의 완료!")

✅ Supervisor 클래스 정의 완료!


## 🔗 Part 7: 노드(Node) 함수 정의

LangGraph에서 사용할 각 에이전트의 노드 함수를 정의합니다.

In [13]:
def supervisor_node(state: MultiAgentState) -> MultiAgentState:
    """감독관 노드 - 작업 분배와 조율"""

    supervisor = Supervisor()

    # 첫 번째 반복: 작업 분석
    if state["iteration"] == 0:
        agent_sequence = supervisor.analyze_task(state["task"])
        state["task_queue"] = [
            {"agent": agent, "status": "pending"} for agent in agent_sequence
        ]
        state["messages"].append(
            AIMessage(
                content=f"Supervisor: 작업 분석 완료. 실행 계획: {' → '.join(agent_sequence)}"
            )
        )

    # 다음 작업 결정
    pending_tasks = [t for t in state["task_queue"] if t["status"] == "pending"]

    if pending_tasks:
        next_agent = pending_tasks[0]["agent"]
        state["current_agent"] = next_agent
        print(f"\n👔 Supervisor: {next_agent}에게 작업 할당")
    else:
        # 모든 작업 완료 - 결과 통합
        if not state["final_output"]:
            final_output = supervisor.integrate_results(
                state["agent_results"], state["task"]
            )
            summary = supervisor.generate_summary(state)
            state["final_output"] = f"{summary}\n\n{final_output}"
            state["messages"].append(
                AIMessage(content="Supervisor: 모든 작업 완료 및 통합 완료! ✨")
            )

    state["iteration"] += 1
    return state


def research_agent_node(state: MultiAgentState) -> MultiAgentState:
    """연구원 에이전트 노드"""

    agent = ResearchAgent()

    # 연구 수행
    result = agent.perform_research(state["task"])

    # 성찰 메커니즘 적용
    improved_result = reflect_and_improve(agent, result, state)

    # 결과 저장
    state["agent_results"][agent.name] = improved_result

    # 작업 큐 업데이트
    for task in state["task_queue"]:
        if task["agent"] == agent.name and task["status"] == "pending":
            task["status"] = "completed"
            break

    # 메시지 추가
    score = state["quality_scores"].get(agent.name, 0)
    state["messages"].append(
        AIMessage(content=f"✅ {agent.name} 완료 (점수: {score:.1f}/10)")
    )

    state["current_agent"] = ""
    return state


def writer_agent_node(state: MultiAgentState) -> MultiAgentState:
    """작가 에이전트 노드"""

    agent = WriterAgent()

    # 연구 데이터 확인
    research_data = state["agent_results"].get("Research Agent")

    # 글 작성
    result = agent.write_content(state["task"], research_data)

    # 성찰 메커니즘 적용
    improved_result = reflect_and_improve(agent, result, state)

    # 결과 저장
    state["agent_results"][agent.name] = improved_result

    # 작업 큐 업데이트
    for task in state["task_queue"]:
        if task["agent"] == agent.name and task["status"] == "pending":
            task["status"] = "completed"
            break

    # 메시지 추가
    score = state["quality_scores"].get(agent.name, 0)
    state["messages"].append(
        AIMessage(content=f"✅ {agent.name} 완료 (점수: {score:.1f}/10)")
    )

    state["current_agent"] = ""
    return state


def critic_agent_node(state: MultiAgentState) -> MultiAgentState:
    """비평가 에이전트 노드"""

    agent = CriticAgent()

    # 평가할 콘텐츠 찾기
    latest_content = None
    for key in ["Writer Agent", "Code Agent", "Research Agent"]:
        if key in state["agent_results"]:
            latest_content = state["agent_results"][key]
            break

    if latest_content:
        # 평가 수행
        evaluation = agent.evaluate(latest_content, state["task"])

        # 결과 저장
        state["agent_results"][agent.name] = evaluation

        # 피드백 생성
        feedback = agent.generate_feedback(evaluation["scores"])

        # 낮은 점수면 다시 작업 큐에 추가할 수 있음
        avg_score = evaluation["average_score"]
        if avg_score < 6.0:
            print(
                f"  ⚠️ 평균 점수가 낮습니다 ({avg_score:.1f}/10). 재작업이 필요할 수 있습니다."
            )

    # 작업 큐 업데이트
    for task in state["task_queue"]:
        if task["agent"] == agent.name and task["status"] == "pending":
            task["status"] = "completed"
            break

    # 메시지 추가
    state["messages"].append(AIMessage(content=f"✅ {agent.name} 평가 완료"))

    state["current_agent"] = ""
    return state


def code_agent_node(state: MultiAgentState) -> MultiAgentState:
    """코드 에이전트 노드"""

    agent = CodeAgent()

    # 컨텍스트 확인
    context = state["agent_results"].get("Research Agent")

    # 코드 생성
    result = agent.generate_code(state["task"], context)

    # 성찰 메커니즘 적용
    improved_result = reflect_and_improve(agent, result, state)

    # 결과 저장
    state["agent_results"][agent.name] = improved_result

    # 작업 큐 업데이트
    for task in state["task_queue"]:
        if task["agent"] == agent.name and task["status"] == "pending":
            task["status"] = "completed"
            break

    # 메시지 추가
    score = state["quality_scores"].get(agent.name, 0)
    state["messages"].append(
        AIMessage(content=f"✅ {agent.name} 완료 (점수: {score:.1f}/10)")
    )

    state["current_agent"] = ""
    return state


print("✅ 모든 노드 함수가 정의되었습니다!")

✅ 모든 노드 함수가 정의되었습니다!


## 🏗️ Part 8: 다중 에이전트 그래프 구성

LangGraph를 사용하여 전체 시스템을 구성합니다.

In [14]:
def should_continue(state: MultiAgentState) -> str:
    """다음 단계를 결정하는 조건부 함수"""

    # 최대 반복 횟수 체크
    if state["iteration"] >= state["max_iterations"]:
        print("  ⚠️ 최대 반복 횟수 도달 - 종료")
        return "end"

    # 최종 출력이 있으면 종료
    if state["final_output"]:
        print("  ✅ 최종 결과 준비 완료 - 종료")
        return "end"

    # 현재 에이전트가 설정되어 있으면 해당 에이전트 실행
    if state["current_agent"]:
        agent_map = {
            "Research Agent": "research",
            "Writer Agent": "writer",
            "Code Agent": "code",
            "Critic Agent": "critic",
        }
        return agent_map.get(state["current_agent"], "supervisor")

    return "supervisor"


def create_multi_agent_system():
    """
    🏗️ 전체 다중 에이전트 시스템 생성
    """

    print("\n🏗️ 다중 에이전트 시스템 구축 중...")

    # 그래프 초기화
    workflow = StateGraph(MultiAgentState)

    # 노드 추가
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("research", research_agent_node)
    workflow.add_node("writer", writer_agent_node)
    workflow.add_node("code", code_agent_node)
    workflow.add_node("critic", critic_agent_node)

    # 조건부 엣지 설정
    workflow.add_conditional_edges(
        "supervisor",
        should_continue,
        {
            "research": "research",
            "writer": "writer",
            "code": "code",
            "critic": "critic",
            "end": END,
        },
    )

    # 각 에이전트에서 다시 supervisor로
    workflow.add_edge("research", "supervisor")
    workflow.add_edge("writer", "supervisor")
    workflow.add_edge("code", "supervisor")
    workflow.add_edge("critic", "supervisor")

    # 시작점 설정
    workflow.set_entry_point("supervisor")

    # 컴파일
    compiled_workflow = workflow.compile()

    print("✅ 다중 에이전트 시스템 구축 완료!")

    return compiled_workflow


# 시스템 생성
multi_agent_system = create_multi_agent_system()


🏗️ 다중 에이전트 시스템 구축 중...
✅ 다중 에이전트 시스템 구축 완료!


## 🧪 Part 9: 테스트 시나리오 실행

다양한 시나리오로 시스템을 테스트합니다.

In [15]:
def run_scenario(scenario_name: str, task: str, system):
    """시나리오 실행 헬퍼 함수"""

    print(f"\n{'='*60}")
    print(f"🎬 시나리오: {scenario_name}")
    print(f"📝 작업: {task}")
    print(f"{'='*60}")

    # 초기 상태 설정
    initial_state = {
        "messages": [HumanMessage(content=task)],
        "current_agent": "",
        "task": task,
        "task_queue": [],
        "agent_results": {},
        "reflection_count": {},
        "quality_scores": {},
        "final_output": None,
        "iteration": 0,
        "max_iterations": 15,
    }

    # 시스템 실행
    final_state = system.invoke(initial_state)

    # 결과 출력
    print("\n" + "=" * 60)
    print("📋 최종 결과:")
    print("=" * 60)

    if final_state["final_output"]:
        # 결과를 짧게 표시 (처음 1500자)
        output = final_state["final_output"]
        if len(output) > 1500:
            print(output[:1500] + "\n\n... [결과가 길어 일부만 표시]")
        else:
            print(output)

    return final_state


print("✅ 시나리오 실행 함수 준비 완료!")

✅ 시나리오 실행 함수 준비 완료!


### 📝 시나리오 1: 단일 에이전트 + 성찰

In [16]:
# 시나리오 1: 단일 에이전트 + 성찰
scenario1_result = run_scenario(
    "단일 에이전트 + 성찰", "AI 윤리에 대한 300자 에세이 작성", multi_agent_system
)


🎬 시나리오: 단일 에이전트 + 성찰
📝 작업: AI 윤리에 대한 300자 에세이 작성

👔 Supervisor: 작업 분석 중...
  📋 선택된 에이전트 순서: Research Agent → Writer Agent → Critic Agent

👔 Supervisor: Research Agent에게 작업 할당

🔍 Research Agent 작업 시작...
  검색 중: AI 윤리에 대한 300자 에세이 작성 2024 2025 latest trends


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = 

  📊 Research Agent 자가 평가 점수: 5.0/10
  🔧 Research Agent 개선 중... (시도 1/3)


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2

  📊 Research Agent 자가 평가 점수: 7.0/10
  ✅ Research Agent: 품질 기준 통과!

👔 Supervisor: Writer Agent에게 작업 할당

✍️ Writer Agent 작업 시작...


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2

  📊 Writer Agent 자가 평가 점수: 7.0/10
  ✅ Writer Agent: 품질 기준 통과!

👔 Supervisor: Critic Agent에게 작업 할당

🎯 Critic Agent 평가 시작...


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2


👔 Supervisor: 결과 통합 중...
  ✅ 최종 결과 준비 완료 - 종료

📋 최종 결과:

        📊 작업 수행 요약
        작업: AI 윤리에 대한 300자 에세이 작성
        참여 에이전트: 3개
        총 반복 횟수: 3회

        품질 점수:
        
  • Research Agent: 7.0/10 (개선 1회)
  • Writer Agent: 7.0/10 (개선 0회)

평균 점수: 7.0/10

AI 윤리는 기술의 이익과 사회적 피해를 균형시키는 과제다. 대형 언어모델은 확률적 패턴 학습기로 오류와 편향, 허위 생성의 위험이 존재한다. 실무적 대응으로 모델카드·데이터시트 공개, 워터마크·출처표시, 사용자 삭제권, 감시·거버넌스, 탐지기 도입이 필요하다. 투명성·책임성·통제가 핵심이다. 예컨대 편향된 채용 알고리즘은 특정 집단을 배제하고, 생성 AI는 딥페이크로 정치·사기·허위정보 유통에 악용된다. 따라서 기업과 정부는 실무 체크리스트를 즉시 적용하고 최신 연구·정책을 근거로 지속해서 개선·감시해야 한다. 함께 협력해야 한다.


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2

### 📝 시나리오 2: 순차적 협업

In [17]:
# 시나리오 2: 순차적 협업
scenario2_result = run_scenario(
    "순차적 협업", "2025년 LLM 트렌드 분석 보고서 작성", multi_agent_system
)

c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



🎬 시나리오: 순차적 협업
📝 작업: 2025년 LLM 트렌드 분석 보고서 작성

👔 Supervisor: 작업 분석 중...
  📋 선택된 에이전트 순서: Research Agent → Writer Agent → Critic Agent → Writer Agent

👔 Supervisor: Research Agent에게 작업 할당

🔍 Research Agent 작업 시작...
  검색 중: 2025년 LLM 트렌드 분석 보고서 작성 2024 2025 latest trends


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2

  📊 Research Agent 자가 평가 점수: 7.0/10
  ✅ Research Agent: 품질 기준 통과!

👔 Supervisor: Writer Agent에게 작업 할당

✍️ Writer Agent 작업 시작...


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2

  📊 Writer Agent 자가 평가 점수: 7.0/10
  ✅ Writer Agent: 품질 기준 통과!

👔 Supervisor: Critic Agent에게 작업 할당

🎯 Critic Agent 평가 시작...


c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:289: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if k in self.model_fields and self.model_fields[k].exclude:
c:\Users\hfdt\AppData\Local\pypoetry\Cache\virtualenvs\jian-AiB0D9CM-py3.11\Lib\site-packages\langchain_core\load\serializable.py:213: PydanticDeprecatedSince2


👔 Supervisor: Writer Agent에게 작업 할당

✍️ Writer Agent 작업 시작...


KeyboardInterrupt: 

### 📝 시나리오 3: 병렬 협업 (코드 포함)

In [ ]:
# 시나리오 3: 병렬 협업 (코드 포함)
scenario3_result = run_scenario(
    "병렬 협업", "초보자를 위한 머신러닝 튜토리얼과 예제 코드 작성", multi_agent_system
)

### 📝 시나리오 4: 복잡한 반복 개선

In [ ]:
# 시나리오 4: 복잡한 반복 개선
scenario4_result = run_scenario(
    "복잡한 반복 개선", "AI 스타트업 비즈니스 제안서 작성", multi_agent_system
)

## 📊 Part 10: 결과 분석 및 시각화

In [ ]:
def analyze_results(scenario_results: Dict):
    """시나리오 결과 분석"""

    print("\n" + "=" * 60)
    print("📊 시나리오 결과 분석")
    print("=" * 60)

    # 품질 점수 분석
    if scenario_results["quality_scores"]:
        print("\n📈 품질 점수:")
        for agent, score in scenario_results["quality_scores"].items():
            bar = "█" * int(score)
            print(f"  {agent:20} {bar} {score:.1f}/10")

        avg_score = sum(scenario_results["quality_scores"].values()) / len(
            scenario_results["quality_scores"]
        )
        print(f"\n  평균 점수: {avg_score:.1f}/10")

    # 성찰 횟수 분석
    if scenario_results["reflection_count"]:
        print("\n🔄 성찰 횟수:")
        for agent, count in scenario_results["reflection_count"].items():
            print(f"  {agent:20} {count}회 개선")

    # 작업 큐 상태
    print("\n✅ 작업 완료 상태:")
    for task in scenario_results["task_queue"]:
        status_icon = "✅" if task["status"] == "completed" else "⏳"
        print(f"  {status_icon} {task['agent']}")


# 시나리오 1 결과 분석
if "scenario1_result" in locals():
    analyze_results(scenario1_result)